In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

/opt/anaconda3/envs/bayes_ml_1/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:
print(f"PyMC Version: {pm.__version__}")

PyMC Version: 5.28.4


In [3]:
# Verify file reading structures
try:
    polaris = pd.read_csv("01_polaris_hotline_signals_fy2013_2024.csv")
    control = pd.read_csv("04_ctdc_means_of_control_by_exploitation.csv")
    print(f"[SUCCESS] Environment verified. Polaris shape: {polaris.shape}, Control matrix shape: {control.shape}")
except FileNotFoundError as e:
    print(f"[ERROR] Could not find the CSV files. Ensure they are in your working directory. Detail: {e}")

[SUCCESS] Environment verified. Polaris shape: (12, 5), Control matrix shape: (32, 6)


In [4]:
##STEP1: State-Space Decomposition Model for Human Trafficking Surveillance.

In [5]:
class BayesianStateSpaceDecomp:
    def __init__(self, data_path: str):
        self.df = pd.read_csv(data_path)
        self.years = self.df['fiscal_year'].values
        self.n_periods = len(self.years)
        
        # Inside the class scope, self is now a valid namespace identifier
        self.y_obs = self.df['potential_situations_identified'].values.astype(float)
        self.x_intervention = (self.years >= 2018).astype(float)
        print("[SUCCESS] Initialized data matrices inside class instance.")

# --- HOW TO RUN IT BELOW THE CLASS ---
# Instantiate an instance named 'decomposer' (this replaces 'self' behind the scenes)
decomposer = BayesianStateSpaceDecomp(data_path="01_polaris_hotline_signals_fy2013_2024.csv")

# Now you can inspect the vectors outside the class using the object's name:
print("Inspected Years Array:", decomposer.years)
print("Observed Vectors Target:", decomposer.y_obs)

[SUCCESS] Initialized data matrices inside class instance.
Inspected Years Array: [2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
Observed Vectors Target: [ 4796.  5166.  5418.  7405.  8686. 10658. 11852. 11193. 10983. 10013.
  9877. 12130.]


In [6]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor          # Crucial: Import base pytensor for access to pytensor.scan
import pytensor.tensor as pt
import arviz as az

class BayesianStateSpaceDecomp:
    def __init__(self, data_path: str):
        """Initializes data matrices inside the class instance context."""
        self.df = pd.read_csv(data_path)
        self.years = self.df['fiscal_year'].values
        self.n_periods = len(self.years)
        self.y_obs = self.df['potential_situations_identified'].values.astype(float)
        self.x_intervention = (self.years >= 2018).astype(float)
        
    def execute_mcmc_sampling(self, draws: int = 1000, tune: int = 1000, target_accept: float = 0.95):
        """Compiles the Dynamic Linear Model graph and runs the NUTS sampler."""
        print("[INFO] Constructing PyMC state-space computational graph...")
        with pm.Model() as self.model:
            # Scale Priors
            sigma_obs = pm.HalfNormal("sigma_obs", sigma=2000)
            sigma_level = pm.HalfNormal("sigma_level", sigma=1500)
            sigma_drift = pm.HalfNormal("sigma_drift", sigma=500)
            beta_fosta = pm.Normal("beta_fosta", mu=0, sigma=4000)
            
            # Initial Latent States
            mu_init = pm.Normal("mu_init", mu=self.y_obs[0], sigma=2000)
            delta_init = pm.Normal("delta_init", mu=0, sigma=500)
            
            # System Innovations (Non-centered parameterization)
            level_innovations = pm.Normal("level_innovations", mu=0, sigma=1, shape=self.n_periods - 1)
            drift_innovations = pm.Normal("drift_innovations", mu=0, sigma=1, shape=self.n_periods - 1)
            
            # State Transition Function for PyTensor Scan
            def state_transition_step(level_inn, drift_inn, prev_level, prev_drift, s_level, s_drift):
                next_drift = prev_drift + drift_inn * s_drift
                next_level = prev_level + next_drift + level_inn * s_level
                return next_level, next_drift

            # FIX: Swapped pt.scan for pytensor.scan to match PyTensor v5 specifications
            states, _ = pytensor.scan(
                fn=state_transition_step,
                sequences=[level_innovations, drift_innovations],
                outputs_info=[mu_init, delta_init],
                non_sequences=[sigma_level, sigma_drift],
                n_steps=self.n_periods - 1
            )
            
            mu_states = pt.concatenate([[mu_init], states[0]])
            pm.Deterministic("latent_trend", mu_states)
            policy_impact = pm.Deterministic("policy_impact", beta_fosta * self.x_intervention)
            
            # Observation Likelihood
            expected_value = mu_states + policy_impact
            pm.Normal("y_likelihood", mu=expected_value, sigma=sigma_obs, observed=self.y_obs)
            
            print("[INFO] Initiating NUTS sampling sequence...")
            self.idata = pm.sample(draws=draws, tune=tune, target_accept=target_accept, random_seed=42)
            
        return self.idata

    def evaluate_diagnostics(self):
        """Evaluates convergence using R-hat."""
        print("\n=== BAYESIAN MCMC CONVERGENCE DIAGNOSTICS ===")
        summary = az.summary(self.idata, var_names=["beta_fosta", "sigma_obs", "sigma_level"])
        print(summary)
        
    def export_residual_signal(self) -> pd.DataFrame:
        """Extracts the filtered demand signature by stripping trends and policy jumps."""
        post_trend = self.idata.posterior["latent_trend"].mean(dim=("chain", "draw")).values
        post_impact = self.idata.posterior["policy_impact"].mean(dim=("chain", "draw")).values
        residual_signal = self.y_obs - (post_trend + post_impact)
        
        return pd.DataFrame({
            "fiscal_year": self.years,
            "observed_situations": self.y_obs,
            "estimated_base_trend": post_trend,
            "policy_shock_component": post_impact,
            "filtered_demand_residual": residual_signal
        })

# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    decomposer = BayesianStateSpaceDecomp(data_path="01_polaris_hotline_signals_fy2013_2024.csv")
    idata = decomposer.execute_mcmc_sampling()
    decomposer.evaluate_diagnostics()
    
    results = decomposer.export_residual_signal()
    print("\n--- SAMPLE EXTRACTED STRUCTURAL RESIDUALS ---")
    print(results.tail(4))

[INFO] Constructing PyMC state-space computational graph...
[INFO] Initiating NUTS sampling sequence...


/var/folders/ps/rfdd1r114171fx_k_r00x8kr0000gn/T/ipykernel_54708/1150416311.py:42: DeprecationWarning: Scan return signature will change. Updates dict will not be returned, only the first argument. Pass `return_updates=False` to conform to the new API and avoid this warning
  states, _ = pytensor.scan(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_obs, sigma_level, sigma_drift, beta_fosta, mu_init, delta_init, level_innovations, drift_innovations]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 247 seconds.
There were 247 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



=== BAYESIAN MCMC CONVERGENCE DIAGNOSTICS ===
                 mean        sd   hdi_3%   hdi_97%  mcse_mean  mcse_sd  \
beta_fosta   1545.063  1333.250 -881.627  4123.787     33.511   26.398   
sigma_obs     512.593   331.150   61.165  1077.388     18.564   11.815   
sigma_level   891.770   431.257   55.266  1629.813     18.121    7.127   

             ess_bulk  ess_tail  r_hat  
beta_fosta     1551.0    1442.0   1.00  
sigma_obs       225.0     126.0   1.01  
sigma_level     543.0    1316.0   1.01  

--- SAMPLE EXTRACTED STRUCTURAL RESIDUALS ---
    fiscal_year  observed_situations  estimated_base_trend  \
8          2021              10983.0           9361.042468   
9          2022              10013.0           8694.307246   
10         2023               9877.0           8728.837619   
11         2024              12130.0          10271.824019   

    policy_shock_component  filtered_demand_residual  
8              1545.062652                 76.894880  
9              1545.0626

In [18]:
"""
Hierarchical Beta-Binomial Pooling Model for Trafficking Means of Control.
"""

class HierarchicalBetaBinomial:
    def __init__(self, data_path: str):
        """Initializes model, handles categorical indices, and prepares tensors."""
        self.df = pd.read_csv(data_path)
        
        # Parse data vectors from the curated 04_ctdc matrix
        self.n_positive = self.df['n_positive'].values.astype(int)
        self.n_observed = self.df['n_observed'].values.astype(int)
        
        # Factorize exploitation types into clean integer category codes (e.g., Labor=0, Sex=1)
        self.df['exploit_code'] = pd.Categorical(self.df['exploitation_type']).codes
        self.exploit_idx = self.df['exploit_code'].values
        self.exploit_labels = pd.Categorical(self.df['exploitation_type']).categories.tolist()
        self.n_groups = len(self.exploit_labels)
        
        # Preserve control tracking names for reporting mappings
        self.means_of_control = self.df['means_of_control'].values
        
    def build_and_compile_model(self):
        """Builds the reparameterized hierarchical Beta-Binomial graph."""
        print("[INFO] Building Hierarchical Beta-Binomial computational graph...")
        
        with pm.Model() as self.model:
            # --- 1. Global Hyper-Prior Layer ---
            # mu: Global base mean prevalence per exploitation category type
            mu = pm.Beta("mu", alpha=1, beta=1, shape=self.n_groups)
            
            # kappa: Sample concentration parameter capturing intra-group variance
            kappa = pm.Pareto("kappa", m=1, alpha=1.5, shape=self.n_groups)
            
            # --- 2. Transforming Hyper-parameters to Beta Shape Constraints ---
            # Leverage indexing array mappings to match scale dimensions dynamically
            alpha_param = pm.Deterministic("alpha_param", mu[self.exploit_idx] * kappa[self.exploit_idx])
            beta_param = pm.Deterministic("beta_param", (1.0 - mu[self.exploit_idx]) * kappa[self.exploit_idx])
            
            # --- 3. Latent Prevalence Parameter (Theta) ---
            # Every distinct mean-of-control row draws from its group-specific Beta matrix
            theta = pm.Beta("theta", alpha=alpha_param, beta=beta_param, shape=len(self.n_positive))
            
            # --- 4. Binomial Observation Likelihood Layer ---
            pm.Binomial(
                "y_likelihood",
                n=self.n_observed,
                p=theta,
                observed=self.n_positive
            )
            
        print("[INFO] Hierarchical model graph compiled successfully.")
        return self.model

    def execute_mcmc_sampling(self, draws: int = 1500, tune: int = 1000):
        """Samples the joint posterior using the standard NUTS sampler setup."""
        if not hasattr(self, 'model'):
            self.build_and_compile_model()
            
        print(f"[INFO] Running NUTS: {draws} draws, {tune} tuning loops...")
        with self.model:
            self.idata = pm.sample(
                draws=draws,
                tune=tune,
                random_seed=42,
                return_inferencedata=True
            )
        print("[INFO] Sampling phase complete.")
        return self.idata

    def generate_prevalence_summary(self) -> pd.DataFrame:
        """Extracts posterior expectations alongside original raw rates."""
        post_theta_means = self.idata.posterior["theta"].mean(dim=("chain", "draw")).values
        
        # Calculate Highest Density Intervals (HDI) for error bounding
        hdi = az.hdi(self.idata, var_names=["theta"])["theta"].values
        
        summary_df = pd.DataFrame({
            "exploitation_type": self.df['exploitation_type'],
            "means_of_control": self.means_of_control,
            "raw_positive": self.n_positive,
            "raw_observed": self.n_observed,
            "empirical_rate": self.n_positive / np.where(self.n_observed == 0, 1, self.n_observed),
            "posterior_pooled_mean": post_theta_means,
            "hdi_3_pct": hdi[:, 0],
            "hdi_97_pct": hdi[:, 1]
        })
        return summary_df

if __name__ == "__main__":
    # Standard local execution verification script
    pooling_engine = HierarchicalBetaBinomial(data_path="04_ctdc_means_of_control_by_exploitation.csv")
    trace = pooling_engine.execute_mcmc_sampling()
    
    summary = pooling_engine.generate_prevalence_summary()
    print("\n=== SAMPLE EXTRACTED POOLED PREVALENCES ===")
    print(summary[['exploitation_type', 'means_of_control', 'empirical_rate', 'posterior_pooled_mean']].head(6))

Initializing NUTS using jitter+adapt_diag...


[INFO] Building Hierarchical Beta-Binomial computational graph...
[INFO] Hierarchical model graph compiled successfully.
[INFO] Running NUTS: 1500 draws, 1000 tuning loops...


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu, kappa, theta]


Output()

Sampling 4 chains for 1_000 tune and 1_500 draw iterations (4_000 + 6_000 draws total) took 3 seconds.
There were 11 divergences after tuning. Increase `target_accept` or reparameterize.


[INFO] Sampling phase complete.

=== SAMPLE EXTRACTED POOLED PREVALENCES ===
  exploitation_type            means_of_control  empirical_rate  \
0     forced_labour                debt_bondage        0.896503   
1     forced_labour              takes_earnings        0.961024   
2     forced_labour  restricts_financial_access        0.000000   
3     forced_labour                     threats        0.969188   
4     forced_labour         psychological_abuse        0.966235   
5     forced_labour              physical_abuse        0.939201   

   posterior_pooled_mean  
0               0.896285  
1               0.960858  
2               0.012432  
3               0.969000  
4               0.965996  
5               0.938910  


In [21]:
"""
Bayesian Classification Utility Maximization & Resource Allocation Framework.
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize

class BayesianResourceOptimizer:
    def __init__(self, demand_posterior_array: np.ndarray, ovc_capacity_path: str):
        """
        Initializes the allocation optimizer context.
        """
        self.demand_samples = demand_posterior_array
        
        # FIX: Changed 'demand_samples.shape' to 'demand_posterior_array.shape' 
        # to correctly reference the local incoming parameter scope.
        self.n_draws, self.n_categories = demand_posterior_array.shape
        
        # Load real-world historical baseline capacity indicators
        self.ovc_df = pd.read_csv(ovc_capacity_path)
        
        # Extract mean baseline quarterly capacities for alignment
        self.baseline_sex_capacity = self.ovc_df['sex_trafficking'].mean()
        self.baseline_labor_capacity = self.ovc_df['labor_trafficking'].mean()
        
    def linex_loss_objective(self, allocation: np.ndarray, asymmetry_params: np.ndarray, scale_costs: np.ndarray) -> float:
        """
        Computes the expected LINEX loss across all sampled MCMC scenarios.
        """
        total_expected_loss = 0.0
        
        for c in range(self.n_categories):
            a = asymmetry_params[c]
            b = scale_costs[c]
            S = allocation[c]
            y_star = self.demand_samples[:, c]
            
            # Loss = b * (exp(a * (y* - S)) - a * (y* - S) - 1)
            error_vector = y_star - S
            loss_vector = b * (np.exp(a * error_vector) - a * error_vector - 1)
            
            # Compute empirical mean across the posterior distribution surface
            total_expected_loss += np.mean(loss_vector)
            
        return total_expected_loss

    def optimize_allocation(self, budget_limit: float, unit_costs: np.ndarray, asymmetry_params: np.ndarray) -> np.ndarray:
        """
        Executes constrained numerical optimization to pinpoint the utility-maximizing capacity.
        """
        print(f"[INFO] Initializing sequential least squares programming optimizer (Hard Budget: ${budget_limit:,})...")
        
        # Starting point for optimization: distribute budget evenly
        initial_guess = np.ones(self.n_categories) * (budget_limit / (self.n_categories * np.mean(unit_costs)))
        
        # Define the linear constraint boundary matrix: Sum(Unit_Cost_c * S_c) <= Budget
        budget_constraint = {
            'type': 'ineq', 
            'fun': lambda S: budget_limit - np.sum(unit_costs * S)
        }
        
        # Enforce non-negativity boundary limits
        bounds = [(0, None) for _ in range(self.n_categories)]
        
        # Run optimization optimization routine
        optimization_result = minimize(
            fun=self.linex_loss_objective,
            x0=initial_guess,
            args=(asymmetry_params, unit_costs),
            method='SLSQP',
            constraints=[budget_constraint],
            bounds=bounds
        )
        
        if not optimization_result.success:
            print("[WARNING] Optimization solver failed to reach optimal termination threshold.")
            
        print("[INFO] Asymmetric utility optimization complete.")
        return optimization_result.x

# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    np.random.seed(42)
    simulated_draws = 2000
    
    # Simulate posterior expected demand for true situations: [Sex_Cases, Labor_Cases]
    simulated_posterior_demand = np.hstack([
        np.random.normal(loc=11000, scale=1200, size=(simulated_draws, 1)), # Sex True Demand
        np.random.normal(loc=4500, scale=600, size=(simulated_draws, 1))   # Labor True Demand
    ])
    
    # Initialize optimization pipeline
    optimizer = BayesianResourceOptimizer(
        demand_posterior_array=simulated_posterior_demand,
        ovc_capacity_path="02_ovc_victims_served_fy23_fy24.csv"
    )
    
    # Configure operational parameters
    asymmetry_parameters = np.array([0.0002, 0.0006])
    estimated_unit_costs = np.array([150.0, 220.0]) 
    available_budget = 2500000.0                   
    
    # Compute optimal allocations
    optimal_capacities = optimizer.optimize_allocation(
        budget_limit=available_budget,
        unit_costs=estimated_unit_costs,
        asymmetry_params=asymmetry_parameters
    )
    
    print("\n=== OPTIMAL UTILITY RESOURCE DEPLOYMENT DEVIATION ===")
    print(f"Optimal Sex Reintegration Capacity Allocation:  {optimal_capacities[0]:.2f} units")
    print(f"Optimal Labor Reintegration Capacity Allocation: {optimal_capacities[1]:.2f} units")
    print(f"Total Budget Expended: ${np.sum(optimal_capacities * estimated_unit_costs):,.2f}")

[INFO] Initializing sequential least squares programming optimizer (Hard Budget: $2,500,000.0)...
[INFO] Asymmetric utility optimization complete.

=== OPTIMAL UTILITY RESOURCE DEPLOYMENT DEVIATION ===
Optimal Sex Reintegration Capacity Allocation:  10103.07 units
Optimal Labor Reintegration Capacity Allocation: 4475.18 units
Total Budget Expended: $2,500,000.00
